In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
import lightgbm as lgb

# Suppress LightGBM warnings
logging.getLogger('lightgbm').setLevel(logging.ERROR)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import optuna


In [ ]:
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('tested.csv')
train = train_df.copy()
test= test_df.copy()
print(f"Training data shape: {train.shape}")
print(f"Testing data shape: {test.shape}")

In [ ]:
train_df.info()

In [ ]:
# Display first few rows of training data
print("First five rows of the training data:")
display(train.head())

# Display information about the dataset
print("\nDataset Information:")
train.info()

# Check for missing values
print("\nMissing values in training data:")
display(train.isnull().sum())

# Statistical summary
print("\nStatistical summary of numeric features:")
display(train.describe())

In [ ]:
train['Name'].nunique()

In [ ]:
train.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
test.drop(columns= ['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace= True)

In [ ]:
train['Age'].fillna(train['Age'].median(), inplace=True)
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)

In [ ]:
print(train.isnull().sum())

In [ ]:
test['Age'].fillna(test['Age'].median(), inplace=True)
test['Fare'].fillna(test['Fare'].median(), inplace=True)

In [ ]:
train['Survived'].value_counts()

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='Survived', data= train)

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='Survived', hue='Sex', data= train)

In [ ]:
sur_p = train.groupby(['Survived', 'Sex']).size().unstack(fill_value=0)
sur_p_percent = sur_p.div(sur_p.sum(axis=1), axis=0) * 100
sur_p_percent.plot(kind='bar', stacked=True, figsize=(8,6))

In [ ]:
sns.histplot(x='Age', data=train, bins=10, color='red')

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='Pclass', y= 'Fare', data= train)

In [ ]:
cols= ['Age', 'SibSp', 'Parch', 'Fare']

train[cols]= train[cols].clip(lower= train[cols].quantile(0.15), upper= train[cols].quantile(0.85), axis=1)

train.drop(columns=['Parch'], axis=1, inplace=True)

In [ ]:
train.select_dtypes(include='number').corr()

In [ ]:
# Correlation heatmap of numeric features
plt.figure(figsize=(10, 8))
numeric_data = train.select_dtypes(include='number')
correlation = numeric_data.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Between Numeric Features')
plt.savefig('correlation_heatmap.png')
plt.show()


In [ ]:
train.plot(kind='box', figsize= (10,8)) 

In [ ]:
test[cols]= test[cols].clip(lower= test[cols].quantile(0.15), upper= test[cols].quantile(0.85), axis=1)

test.drop(columns=['Parch'], axis=1, inplace=True)

In [ ]:
train= pd.get_dummies(train, columns=['Pclass', 'Sex', 'Embarked' ], drop_first= True)

test= pd.get_dummies(test, columns=['Pclass', 'Sex', 'Embarked' ], drop_first= True)

In [ ]:
train.head()

In [ ]:
X_train= train.iloc[:, 1:]
y_train= train.iloc[:,0].values.reshape(-1,1)


In [ ]:
#from sklearn.model_selection import train_test_split

#X_train,X_test,y_train,y_test = train_test_split(X_train,y_train,test_size=0.1)


X_test= test.iloc[:, 1:]
y_test= test.iloc[:,0].values.reshape(-1,1)


In [ ]:
from sklearn.preprocessing import StandardScaler
ss= StandardScaler()
features= ['Age', 'SibSp', 'Fare']

X_train[features]= ss.fit_transform(X_train[features])
X_test[features]= ss.fit_transform(X_test[features])

In [ ]:
from sklearn.linear_model import LogisticRegression

clf= LogisticRegression()

clf.fit(X_train, y_train.ravel())

predictions= clf.predict(X_test)

In [ ]:
print(clf.score(X_train, y_train))

In [ ]:
cross_val_score(clf, X_train, y_train.ravel(), cv= 5).mean()

In [ ]:
# Ensure test_df is defined and available
#submission = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Survived': predictions})

# Display the first few rows of the submission dataframe
#print(submission.head())

In [ ]:
#submission.to_csv('submission.csv', index= False)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1]
}

In [ ]:
import lightgbm
model_cls = lightgbm.LGBMClassifier(verbose=-1)
results = GridSearchCV(estimator=model_cls, param_grid=param_grid, scoring='accuracy', cv=5, verbose=1, n_jobs=-1)

In [ ]:
results.fit(X_train, y_train.ravel())

In [ ]:
display(results.best_params_)

In [ ]:
model = display(results.best_estimator_)

In [ ]:
display(results.best_score_)

In [ ]:

import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Objective function for Optuna
def objective(trial):
    param = {
        "objective": "binary",
        "metric": "binary_error",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "feature_pre_filter": False,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        lgb.LGBMClassifier(**param),
        X_train, y_train,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1
    )

    return 1.0 - scores.mean()  # minimize error (maximize accuracy)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

# Show best trial
print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {1 - trial.value:.4f}")
print("  Best hyperparameters: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
train_df['set'] = 'train'
test_df['set'] = 'test'
all_df = pd.concat([train_df, test_df], axis=0)

In [ ]:
all_df["Family Size"] = all_df["SibSp"] + all_df["Parch"] + 1
all_df["Age Interval"] = 0.0
all_df.loc[ all_df['Age'] <= 16, 'Age Interval']  = 0
all_df.loc[(all_df['Age'] > 16) & (all_df['Age'] <= 32), 'Age Interval'] = 1
all_df.loc[(all_df['Age'] > 32) & (all_df['Age'] <= 48), 'Age Interval'] = 2
all_df.loc[(all_df['Age'] > 48) & (all_df['Age'] <= 64), 'Age Interval'] = 3
all_df.loc[ all_df['Age'] > 64, 'Age Interval'] = 4
all_df['Fare Interval'] = 0.0
all_df.loc[ all_df['Fare'] <= 7.91, 'Fare Interval'] = 0
all_df.loc[(all_df['Fare'] > 7.91) & (all_df['Fare'] <= 14.454), 'Fare Interval'] = 1
all_df.loc[(all_df['Fare'] > 14.454) & (all_df['Fare'] <= 31), 'Fare Interval']   = 2
all_df.loc[ all_df['Fare'] > 31, 'Fare Interval'] = 3
all_df["Sex_Pclass"] = all_df.apply(lambda row: row['Sex'][0].upper() + "_C" + str(row["Pclass"]), axis=1)
def get_deck(text):
    try:
        return text[0]
    except Exception as ex:
        return "Unknown"
    
all_df["Deck"] = all_df["Cabin"].apply(lambda x: get_deck(x))

In [ ]:
def parse_names(row):
    try:
        text = row["Name"]
        split_text = text.split(",")
        family_name = split_text[0]
        next_text = split_text[1]
        split_text = next_text.split(".")
        title = (split_text[0] + ".").lstrip().rstrip()
        next_text = split_text[1]
        if "(" in next_text:
            split_text = next_text.split("(")
            given_name = split_text[0]
            maiden_name = split_text[1].rstrip(")")
            return pd.Series([family_name, title, given_name, maiden_name])
        else:
            given_name = next_text
            return pd.Series([family_name, title, given_name, None])
    except Exception as ex:
        print(f"Exception: {ex}")
    
all_df[["Family Name", "Title", "Given Name", "Maiden Name"]] = all_df.apply(lambda row: parse_names(row), axis=1)

In [ ]:
from wordcloud import WordCloud, STOPWORDS
stopwords = set(STOPWORDS)

def show_wordcloud(data, mask=None, title=""):
    text = " ".join(t for t in data.dropna())
    stopwords = set(STOPWORDS)
    stopwords.update(["t", "co", "https", "amp", "U", "Comment", "text", "attr", "object"])
    wordcloud = WordCloud(stopwords=stopwords, scale=4, max_font_size=50, max_words=500,mask=mask, background_color="white").generate(text)
    fig = plt.figure(1, figsize=(12, 12))
    plt.axis('off')
    fig.suptitle(title, fontsize=14)
    fig.subplots_adjust(top=2.3)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.show()    
show_wordcloud(all_df["Family Name"], title="Family Names on Titanic")

In [ ]:
all_df.head()

In [ ]:
all_df.info()

In [ ]:
# Print number of unique values in each column
print("\nUnique values in each column:")
print(all_df.nunique())

In [ ]:
print(all_df.isnull().sum())

In [ ]:
all_df["Age"].fillna(all_df["Age"].median(), inplace=True)
all_df["Maiden Name"].fillna("Unknown", inplace=True)
all_df["Cabin"].fillna("Unknown", inplace=True)

In [ ]:
#Columns with unique values that are more than 200

unique_counts = all_df.nunique()
unique_columns = unique_counts[unique_counts > 200].index.tolist()
dropped_all_df = all_df.drop(columns=unique_columns + ["set", "Survived"])

cat_columns = dropped_all_df.select_dtypes(include=["object"]).columns.tolist()
dummy_all_df = pd.get_dummies(dropped_all_df, columns=cat_columns, drop_first=True)

In [ ]:

X_train = dummy_all_df[all_df["set"] == "train"]
X_test = dummy_all_df[all_df["set"] == "test"]
y_train = all_df[all_df["set"] == "train"]["Survived"]
y_test = all_df[all_df["set"] == "test"]["Survived"]


#X_train,X_test,y_train,y_test = train_test_split(X_train,y_train,test_size=0.1)




In [ ]:
all_model_results = {}

In [ ]:
from sklearn.linear_model import LogisticRegression

clf= LogisticRegression()

clf.fit(X_train, y_train.ravel())

predictions= clf.predict(X_test)

In [ ]:
X_test

In [ ]:
score = clf.score(X_test, y_test)
all_model_results['LR'] = score
print(f"test score of LR: {score}")

cross_val_score(clf, X_train, y_train, cv= 5).mean()


In [ ]:

model_cls = lightgbm.LGBMClassifier(verbose=-1)
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1]
}
#param_grid = {
#    # Learning rate: Controls the step size. Smaller values usually require more trees.
#    'learning_rate': [0.01, 0.05, 0.1],
#
#    # Number of boosting rounds (trees)
#    'n_estimators': [100, 200, 300],
#
#    # Max number of leaves in one tree. Key parameter for complexity.
#    'num_leaves': [15, 31, 47], # Default is 31. Try lower and slightly higher.
#
#    # Maximum tree depth. -1 means no limit.
#    'max_depth': [-1, 5, 10],
#
#    # Minimum number of data points needed in a child (leaf).
#    # Higher values prevent creating too specific leaves, thus regularizing.
#    'min_child_samples': [10, 20, 30], # Default is 20
#
#    'subsample': [0.7, 0.8, 0.9, 1.0], # Default is 1.0
#
#    # Subsample ratio of columns when constructing each tree. (Feature fraction)
#    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # Default is 1.0
#
#    # L1 regularization term on weights
#    'reg_alpha': [0, 0.01, 0.1],
#
#    # L2 regularization term on weights
#    'reg_lambda': [0, 0.01, 0.1],
#
#}
results = GridSearchCV(estimator=model_cls, param_grid=param_grid, scoring='accuracy', cv=5, verbose=1, n_jobs=-1)


results.fit(X_train, y_train.ravel())

display(results.best_score_)

In [ ]:
model = lightgbm.LGBMClassifier(**results.best_params_,verbose=-1)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
print(f"test score of LightGBM: {model.score(X_test, y_test)}")

In [ ]:
model = lightgbm.LGBMClassifier(verbose=-1)

In [ ]:
model.fit(X_train,y_train)

score = model.score(X_test, y_test)
all_model_results['LightGBM'] = score
print(f"test score of LightGBM: {score}")

In [ ]:
features = X_train.columns
importances = model.feature_importances_
# Get default ('split') importance using the property:
split_importance = model.feature_importances_ 

# Get 'gain' importance by accessing the underlying booster's METHOD:
gain_importance = model.booster_.feature_importance(importance_type='gain')
feature_importance = pd.Series(importances, index=features).sort_values(ascending=False)

In [ ]:
lgb.plot_importance(model, importance_type="gain", figsize=(7,6), title="LightGBM Feature Importance (Gain)")
plt.show()

In [ ]:
lgb.plot_importance(model, importance_type="split",  figsize=(7,6), title="LightGBM Feature Importance (Gain)")
plt.show()

In [ ]:
import shap
import numpy as np  
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

In [ ]:

# 1. Summary Plot (Feature Importance)
shap.summary_plot(shap_values, X_train, plot_type="bar")
# This shows the average impact of each feature on the model output.

# 2. Summary Plot (Detailed, Beeswarm)
shap.summary_plot(shap_values, X_train)
# This shows the distribution of SHAP values for each feature, with colors indicating feature values (e.g., high vs. low).

# 3. Force Plot (Individual Prediction)
# For a single instance (e.g., first row of X_train)
shap.initjs()  # Required for JavaScript-based plots in Jupyter
shap.force_plot(explainer.expected_value, shap_values[0], X_train.iloc[0])
# This visualizes the contribution of each feature to a single prediction.

# 4. Dependence Plot (Feature Interaction)
# For a specific feature (e.g., 'feature_name')
#shap.dependence_plot('feature_name', shap_values, X_train)
# This shows how SHAP values for a feature vary with its values, optionally colored by another feature.

# 5. Decision Plot (Multiple Instances)
# For a subset of instances (e.g., first 100 rows)
shap.decision_plot(explainer.expected_value, shap_values[:100], X_train.iloc[:100])
# This shows how features contribute to predictions for multiple instances.

In [ ]:
shap_values = explainer.shap_values(X_train[:1])

In [ ]:

shap.summary_plot(shap_values, X_train[:1], plot_type="bar")

In [ ]:

shap.summary_plot(shap_values, X_train[:1])

In [ ]:
feature_importance

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train.ravel())

score = model.score(X_test, y_test)
all_model_results['DT'] = score
print("Decision Tree Test Score:", score)   

In [ ]:
from sklearn.tree import export_text

# Assuming 'clf' is your trained decision tree model
rules = export_text(model, feature_names=X_train.columns.tolist())
print("Decision Tree Rules:")
print(rules)

In [ ]:
features = X_train.columns
importances = model.feature_importances_
feature_importance = pd.Series(importances, index=features).sort_values(ascending=False)

In [ ]:
feature_importance

In [ ]:
import shap
import numpy as np  
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

In [ ]:
shap_values.shape

In [ ]:

shap_feature_importance = pd.Series(np.mean(np.abs(shap_values), axis=0)[:,1], index=features).sort_values(ascending=False)

In [ ]:
shap_feature_importance

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier()
model.fit(X_train, y_train.ravel())

score = model.score(X_test, y_test)
all_model_results['CatBoost'] = score
print("XGBoost Test Score:", score)
importances = model.feature_importances_
feature_importance = pd.Series(importances, index=features).sort_values(ascending=False)


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)
shap_values.shape

In [ ]:
shap_feature_importance = pd.Series(np.mean(np.abs(shap_values.values), axis=0), index=features).sort_values(ascending=False)

In [ ]:
shap_feature_importance

In [ ]:
import catboost as cat
from catboost import CatBoostClassifier
model = CatBoostClassifier(verbose=0)
model.fit(X_train, y_train)

score = model.score(X_test, y_test)
all_model_results['CatBoost'] = score
print("CatBoost Test Score:", score)
importances = model.feature_importances_
feature_importance = pd.Series(importances, index=features).sort_values(ascending=False)
feature_importance


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

In [ ]:
shap_feature_importance = pd.Series(np.mean(np.abs(shap_values.values), axis=0), index=features).sort_values(ascending=False)

In [ ]:
shap_feature_importance

In [ ]:
#! pip install git+https://github.com/milesqli/iffnn.git 

# Explanation With My IFFNN

In [ ]:
import torch
from iffnn import IFFNN

model = IFFNN(
    input_size=len(X_train.columns),
    num_classes=1,
    feature_names=X_train.columns.tolist(),  # Optional
    class_names=['Not Survived', 'Survived'],      # Optional
    hidden_sizes=[226, 113, 113,113,226], #None,            # Use default hidden layers
    device='auto'                 # Use CUDA if available
)

In [ ]:
# Convert bool columns to int
X_train_NN_ = X_train.astype({col: 'int' for col in X_train.select_dtypes(include='bool').columns})

In [ ]:
from sklearn.model_selection import train_test_split
X_train_NN,X_valid_NN,y_train_NN,y_valid_NN = train_test_split(X_train_NN_,y_train,test_size=0.05)

In [ ]:
# Convert bool columns to int
X_test_NN = X_test.astype({col: 'int' for col in X_test.select_dtypes(include='bool').columns})

In [ ]:
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train_NN.values, dtype=torch.float32), torch.tensor(y_train_NN.values, dtype=torch.float32)),
    batch_size=64,
    shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_valid_NN.values, dtype=torch.float32), torch.tensor(y_valid_NN.values, dtype=torch.float32)),
    batch_size=64,
    shuffle=True
)


In [ ]:
history = model.train_model(
    train_loader=train_loader,
    valid_loader=valid_loader,
    num_epochs=30, # Adjust as needed
    save_path='best_iffnn_model.pth' # Optional: saves the best model based on validation accuracy
)

In [ ]:
explanations = model.explain(
    X_train_NN.values[:5],   # Your batch of input features
    top_n=5,          # Show top 5 features per class
    print_output=True # Print explanations to console
)

In [ ]:

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_test_NN.values, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.int)),
    batch_size=64,
    shuffle=True
)

In [ ]:
res = model.evaluate_model(test_loader)
score = res['test_accuracy']
all_model_results['IFFNN'] = score/100
print("IFFNN Test Score:", score)

In [ ]:
for mo, acc in all_model_results.items():
    print(f"Accuracy of {mo} is {acc*100:.2f}%")